In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
import random
import json

# Enhanced Dataset
intents = {
    "intents": [
        {"tag": "greeting", "patterns": ["Hi", "Hello", "Hey", "Hi there", "Is anyone there?"],
         "responses": ["Hello!", "Hi there, how can I help?", "Hey! What's on your mind?"]},
        {"tag": "goodbye", "patterns": ["Bye", "See you later", "Goodbye", "I'm leaving"],
         "responses": ["Goodbye!", "Have a great day!", "See you soon!"]},
        {"tag": "identity", "patterns": ["Who are you?", "What are you?", "What is your name?"],
         "responses": ["I am a Deep Learning Chatbot.", "I'm your AI assistant."]},
        {"tag": "capabilities", "patterns": ["What can you do?", "How can you help me?", "What is your purpose?"],
         "responses": ["I can chat with you and answer basic questions based on my training data!"]},
        {"tag": "unknown", "patterns": [], "responses": ["I'm sorry, I don't have information on that yet.", "Could you rephrase that? I'm not sure I understand."]}
    ]
}

In [ ]:
training_sentences = []
training_labels = []
labels = []
responses = []

for intent in intents['intents']:
    for pattern in intent['patterns']:
        training_sentences.append(pattern.lower())
        training_labels.append(intent['tag'])
    responses.append(intent['responses'])
    if intent['tag'] not in labels:
        labels.append(intent['tag'])

num_classes = len(labels)

# Tokenization
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>") # OOV = Out Of Vocabulary
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating='post', maxlen=20)

# Label Encoding
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
enc.fit(training_labels)
training_labels = enc.transform(training_labels)

In [ ]:
model = Sequential([
    Embedding(1000, 16, input_length=20),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(padded_sequences, np.array(training_labels), epochs=500, verbose=0)
print("Advanced Brain Trained.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Advanced Brain Trained.


In [ ]:
def chat():
    print("AI Chatbot v2 (Type 'quit' to exit)")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break

        # Process input
        seq = tokenizer.texts_to_sequences([user_input.lower()])
        padded = pad_sequences(seq, truncating='post', maxlen=20)

        # Predict
        prediction = model.predict(padded, verbose=0)
        confidence = np.max(prediction) # How sure is the bot?
        tag = enc.inverse_transform([np.argmax(prediction)])[0]

        # Confidence Threshold (70%)
        if confidence < 0.70:
            print("Bot: I'm not quite sure about that. Can you try asking differently?")
        else:
            for i in intents['intents']:
                if i['tag'] == tag:
                    print(f"Bot: {random.choice(i['responses'])}")

chat()

AI Chatbot v2 (Type 'quit' to exit)
Bot: Hello!
Bot: I'm your AI assistant.
Bot: I'm your AI assistant.
